# MoTe2 Three-Band NetKet Notebook

This notebook mirrors the Haldane workflow for the MoTe2 three-band model:
1. configure GPU backend
2. inspect single-particle bands and k-space diagnostics
3. build the interacting many-body Hamiltonian in NetKet
4. validate NetKet ED against `bloch_ed` for nonzero interaction
5. run a short `VMC_SR` optimization with either `slater` or `slater_x_boseformer`


In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("Set CUDA_VISIBLE_DEVICES=0. If JAX was already imported in this kernel, restart kernel now.")


In [ ]:
import json
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import netket as nk
import numpy as np
import optax

from aidan_custom import (
    build_mote2_three_orbital_hamiltonian,
    noninteracting_slater_orbitals_mote2_three_orbital,
)
from aidan_custom.bloch_ed import (
    build_fock_basis,
    build_many_body_hamiltonian_dense,
    build_mote2_three_orbital_real_space_terms,
    build_mote2_three_orbital_lattice_embedding,
)
from aidan_custom.models import (
    LogSlaterBoseFormer,
    LogSlaterDeterminant,
    make_supercell_reciprocal_vectors,
)
from aidan_custom.mote2_three_orbital import (
    MOTE2_A1,
    MOTE2_A2,
    mote2_three_orbital_berry_and_metric_trace,
    mote2_three_orbital_eigenvalues,
    mote2_three_orbital_reciprocal_vectors,
)
from aidan_custom.optimization import (
    exact_manybody_ground_state_energy,
    log_optimization_diagnostics,
)


In [ ]:
def require_gpu_backend() -> None:
    # Written with Codex 02-20-26.
    backend = jax.default_backend()
    if backend != "gpu":
        raise RuntimeError(
            "This notebook must run on GPU. "
            f"Found default_backend={backend!r}."
        )


def quartic_terms_to_tensor(site_terms, n_sites: int) -> np.ndarray:
    # Written with Codex 02-20-26.
    two_body = np.zeros((n_sites, n_sites, n_sites, n_sites), dtype=np.complex128)
    n_terms = int(site_terms.coefficients.size)
    for idx in range(n_terms):
        a = int(site_terms.create_1[idx])
        b = int(site_terms.create_2[idx])
        c = int(site_terms.annihilate_1[idx])
        d = int(site_terms.annihilate_2[idx])
        two_body[a, b, c, d] += np.complex128(site_terms.coefficients[idx])
    return two_body


def hashable_matrix(matrix: np.ndarray) -> tuple[tuple[float, ...], ...]:
    # Written with Codex 02-20-26.
    arr = np.asarray(matrix, dtype=np.float64)
    return tuple(tuple(float(v) for v in row) for row in arr.tolist())


def mote2_site_positions(Lx: int, Ly: int, a_m: float) -> np.ndarray:
    # Written with Codex 02-20-26.
    lattice = build_mote2_three_orbital_lattice_embedding(Lx=Lx, Ly=Ly)
    n_sites = int(Lx) * int(Ly) * int(lattice.n_orbitals_per_cell)

    a1 = float(a_m) * np.asarray(MOTE2_A1, dtype=np.float64)
    a2 = float(a_m) * np.asarray(MOTE2_A2, dtype=np.float64)
    u0 = np.asarray([float(a_m) / np.sqrt(3.0), 0.0], dtype=np.float64)
    orbital_offsets = np.asarray([u0, np.zeros(2), -u0], dtype=np.float64)

    pos = np.empty((n_sites, 2), dtype=np.float64)
    for site in range(n_sites):
        x, y, orb = lattice.site_to_cell[site]
        pos[site] = float(x) * a1 + float(y) * a2 + orbital_offsets[int(orb)]
    return pos


In [ ]:
require_gpu_backend()
print("devices:", jax.devices())
print("default_backend:", jax.default_backend())


In [ ]:
params = {
    "Lx": 1,
    "Ly": 2,
    "n_fermions": 2,
    "delta": 0.15,
    "ez": 0.05,
    "t_th1": 1.00,
    "t_hh1": 0.85,
    "t_th2": 0.20,
    "t_hh3": -0.10,
    "t_tt1": 0.07,
    "a_m": 1.0,
    "ph_conj": False,
    "V1": 0.60,
}

model_type = "slater"  # options: "slater", "slater_x_boseformer"

boseformer_num_layers = 2
boseformer_d_model = 32
boseformer_n_heads = 4
boseformer_mlp_hidden_factor = 4

print(params)
print("model_type:", model_type)


In [ ]:
b1, b2 = mote2_three_orbital_reciprocal_vectors(a_m=params["a_m"])

gamma = np.array([0.0, 0.0])
k_point = (2.0 * b1 + b2) / 3.0
m_point = 0.5 * b1
nodes = [gamma, k_point, m_point, gamma]
labels = ["$\Gamma$", "K", "M", "$\Gamma$"]

points_per_segment = 80
k_path = []
x_axis = []
x_nodes = [0.0]
s = 0.0
for i in range(len(nodes) - 1):
    start = nodes[i]
    end = nodes[i + 1]
    seg = np.linspace(0.0, 1.0, points_per_segment, endpoint=False)
    seg_points = start[None, :] * (1.0 - seg[:, None]) + end[None, :] * seg[:, None]
    if i > 0:
        seg_points = seg_points[1:]
    for p in seg_points:
        k_path.append(p)
    segment_length = np.linalg.norm(end - start)
    npts = seg_points.shape[0]
    if npts > 0:
        x_local = np.linspace(s, s + segment_length, npts, endpoint=False)
        x_axis.extend(x_local.tolist())
    s += segment_length
    x_nodes.append(s)

k_path = np.asarray(k_path)
x_axis = np.asarray(x_axis)
bands = np.asarray([
    mote2_three_orbital_eigenvalues(kx, ky, ph_conj=params["ph_conj"],
                                 delta=params["delta"], ez=params["ez"],
                                 t_th1=params["t_th1"], t_hh1=params["t_hh1"],
                                 t_th2=params["t_th2"], t_hh3=params["t_hh3"],
                                 t_tt1=params["t_tt1"], a_m=params["a_m"])
    for kx, ky in k_path
])

fig, ax = plt.subplots(figsize=(7, 4))
for ib in range(3):
    ax.plot(x_axis, bands[:, ib], lw=1.5, color="black")
for xn in x_nodes:
    ax.axvline(xn, color="0.8", lw=0.8)
ax.set_xticks(x_nodes)
ax.set_xticklabels(labels)
ax.set_ylabel("Energy")
ax.set_title("MoTe2 three-band structure")
ax.grid(axis="y", alpha=0.25)
plt.show()

omega_k, tr_g_k = mote2_three_orbital_berry_and_metric_trace(
    k_point[0],
    k_point[1],
    band="lowest",
    ph_conj=params["ph_conj"],
    delta=params["delta"],
    ez=params["ez"],
    t_th1=params["t_th1"],
    t_hh1=params["t_hh1"],
    t_th2=params["t_th2"],
    t_hh3=params["t_hh3"],
    t_tt1=params["t_tt1"],
    a_m=params["a_m"],
)
print(f"K point = {k_point}")
print(f"Berry curvature at K (lowest band) = {omega_k:.8f}")
print(f"Tr[g] at K (lowest band) = {tr_g_k:.8f}")


In [ ]:
graph, hi, ham = build_mote2_three_orbital_hamiltonian(**params)
ham_sr = ham.to_fermionoperator2nd()

print(f"n_nodes = {graph.n_nodes}")
print(f"n_edges = {len(graph.edges())}")
print(f"hilbert.n_states = {hi.n_states}")
print(f"operator type = {type(ham).__name__}")


In [ ]:
e_nk = float(np.real(np.asarray(nk.exact.lanczos_ed(ham, k=1, compute_eigenvectors=False)).ravel()[0]))
spec_nk = np.sort(np.real(np.asarray(nk.exact.full_ed(ham, compute_eigenvectors=False), dtype=np.complex128)))

_, one_body_real_space, site_terms = build_mote2_three_orbital_real_space_terms(
    Lx=params["Lx"],
    Ly=params["Ly"],
    delta=params["delta"],
    ez=params["ez"],
    t_th1=params["t_th1"],
    t_hh1=params["t_hh1"],
    t_th2=params["t_th2"],
    t_hh3=params["t_hh3"],
    t_tt1=params["t_tt1"],
    a_m=params["a_m"],
    ph_conj=params["ph_conj"],
    V1=params["V1"],
)
n_sites = int(one_body_real_space.shape[0])
two_body = quartic_terms_to_tensor(site_terms=site_terms, n_sites=n_sites)
basis = build_fock_basis(n_orbitals=n_sites, n_particles=int(params["n_fermions"]))
h_bloch_dense = build_many_body_hamiltonian_dense(
    one_body=one_body_real_space,
    two_body=two_body,
    basis=basis,
    cutoff=1e-12,
    hermitize=True,
)
spec_bloch = np.linalg.eigvalsh(h_bloch_dense).real
e_bloch = float(spec_bloch[0])

ground_abs_diff = float(abs(e_nk - e_bloch))
spec_max_abs_diff = float(np.max(np.abs(spec_nk - spec_bloch)))

print(f"NetKet ED ground energy = {e_nk:.16f}")
print(f"bloch_ed ground energy = {e_bloch:.16f}")
print(f"|ground diff| = {ground_abs_diff:.3e}")
print(f"max spectrum diff = {spec_max_abs_diff:.3e}")

tol = 1e-10
if not (ground_abs_diff < tol and spec_max_abs_diff < tol):
    raise AssertionError(
        f"ED mismatch above tolerance {tol}: ground={ground_abs_diff}, spectrum={spec_max_abs_diff}"
    )


In [ ]:
n_iter = 300
diag_shift = 0.01

initial_orbitals = noninteracting_slater_orbitals_mote2_three_orbital(
    Lx=params["Lx"],
    Ly=params["Ly"],
    n_fermions=params["n_fermions"],
    delta=params["delta"],
    ez=params["ez"],
    t_th1=params["t_th1"],
    t_hh1=params["t_hh1"],
    t_th2=params["t_th2"],
    t_hh3=params["t_hh3"],
    t_tt1=params["t_tt1"],
    a_m=params["a_m"],
    ph_conj=params["ph_conj"],
)

if model_type == "slater":
    model = LogSlaterDeterminant(
        hilbert=hi,
        param_dtype=jnp.float64,
        split_complex_params=True,
        initial_m_orbitals=initial_orbitals,
    )
elif model_type == "slater_x_boseformer":
    positions = mote2_site_positions(
        Lx=params["Lx"],
        Ly=params["Ly"],
        a_m=params["a_m"],
    )
    basis_vectors = np.stack([
        float(params["a_m"]) * np.asarray(MOTE2_A1, dtype=np.float64),
        float(params["a_m"]) * np.asarray(MOTE2_A2, dtype=np.float64),
    ], axis=0)
    g_vectors = np.asarray(
        make_supercell_reciprocal_vectors(
            basis_vectors=basis_vectors,
            Lx=params["Lx"],
            Ly=params["Ly"],
        ),
        dtype=np.float64,
    )

    model = LogSlaterBoseFormer(
        hilbert=hi,
        positions=hashable_matrix(positions),
        g_vectors=hashable_matrix(g_vectors),
        num_layers=boseformer_num_layers,
        d_model=boseformer_d_model,
        n_heads=boseformer_n_heads,
        mlp_hidden_factor=boseformer_mlp_hidden_factor,
        slater_param_dtype=jnp.float64,
        slater_initial_m_orbitals=initial_orbitals,
        boseformer_param_dtype=jnp.float64,
    )
else:
    raise ValueError(f"Unknown model_type={model_type!r}.")

vstate = nk.vqs.FullSumState(hi, model)

optimizer = nk.optimizer.Sgd(
    learning_rate=optax.linear_schedule(0.05, 0.01, n_iter)
)
driver = nk.driver.VMC_SR(
    ham_sr,
    optimizer,
    variational_state=vstate,
    diag_shift=diag_shift,
    mode="complex",
)

log = nk.logging.RuntimeLog()
driver.run(n_iter=n_iter, out=log, callback=log_optimization_diagnostics)


In [ ]:
energy_hist = log.data["Energy"]
iters = np.asarray(energy_hist.iters)
energy_mean = np.asarray(energy_hist["Mean"])
energy_sigma = np.asarray(energy_hist["Sigma"])

e_exact = exact_manybody_ground_state_energy(ham)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(iters, np.real(energy_mean), lw=1.6, color="tab:blue", label="VMC_SR")
ax.fill_between(
    iters,
    np.real(energy_mean - energy_sigma),
    np.real(energy_mean + energy_sigma),
    color="tab:blue",
    alpha=0.2,
    linewidth=0.0,
)
ax.axhline(e_exact, color="tab:red", linestyle="--", label="Exact ED")
ax.set_xlabel("Iteration")
ax.set_ylabel("Energy")
ax.set_title(f"MoTe2 three-band: {model_type} vs exact ED")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

print(f"Final VMC energy (real) = {np.real(energy_mean[-1]):.12f}")
print(f"Exact ED ground energy = {e_exact:.12f}")


In [ ]:
result_dir = Path("results/mote2_netket") / (
    f"lx{params['Lx']}_ly{params['Ly']}_nf{params['n_fermions']}_"
    f"delta_{params['delta']:.2f}_ez_{params['ez']:.2f}_"
    f"tth1_{params['t_th1']:.2f}_thh1_{params['t_hh1']:.2f}_"
    f"tth2_{params['t_th2']:.2f}_thh3_{params['t_hh3']:.2f}_"
    f"ttt1_{params['t_tt1']:.2f}_V1_{params['V1']:.2f}_"
    f"{model_type}_fullsum_vmc"
)
raw_data_dir = result_dir / "raw_data"
result_dir.mkdir(parents=True, exist_ok=True)
raw_data_dir.mkdir(parents=True, exist_ok=True)

np.savez(
    raw_data_dir / "ed_comparison.npz",
    netket_spectrum=spec_nk,
    bloch_ed_spectrum=spec_bloch,
    vmc_iters=iters,
    vmc_energy_mean=np.asarray(energy_mean),
    vmc_energy_sigma=np.asarray(energy_sigma),
)

log.serialize(result_dir / "runtime_log")
summary = {
    "parameters": params,
    "model_type": model_type,
    "boseformer": {
        "num_layers": boseformer_num_layers,
        "d_model": boseformer_d_model,
        "n_heads": boseformer_n_heads,
        "mlp_hidden_factor": boseformer_mlp_hidden_factor,
    },
    "ed_tolerance": 1e-10,
    "ed_ground_abs_diff": ground_abs_diff,
    "ed_spectrum_max_abs_diff": spec_max_abs_diff,
    "exact_ground_energy": float(e_exact),
    "final_vmc_energy_real": float(np.real(energy_mean[-1])),
}
(result_dir / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")

print(f"Saved results to: {result_dir}")
